# Model Comparison: TF-IDF + Logistic Regression vs. Fine-tuned BERT

This notebook compares the performance of both approaches and provides recommendations.

## Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Imports successful!")

Imports successful!


## Load Metrics from Both Models

In [2]:
# Load metrics
with open('tfidf_metrics.json', 'r') as f:
    tfidf_metrics = json.load(f)

with open('bert_results.json', 'r') as f:
    bert_metrics = json.load(f)

print("TF-IDF Metrics:")
print(json.dumps(tfidf_metrics, indent=2))

print("\nBERT Metrics:")
print(json.dumps(bert_metrics, indent=2))

TF-IDF Metrics:
{
  "model_type": "TF-IDF + Logistic Regression",
  "test_accuracy": 1.0,
  "test_weighted_f1": 1.0,
  "test_macro_f1": 1.0,
  "val_accuracy": 1.0,
  "val_weighted_f1": 1.0,
  "train_accuracy": 1.0,
  "train_weighted_f1": 1.0
}

BERT Metrics:
{
  "model_type": "BERT-Base Fine-tuned",
  "hardware": "Apple M1 8GB",
  "training_time_hours": 0.14512933472792308,
  "test_accuracy": 1.0,
  "test_weighted_f1": 1.0,
  "test_macro_f1": 1.0,
  "val_accuracy": 1.0,
  "val_weighted_f1": 1.0,
  "training_loss": 2.997203017726089,
  "hyperparameters": {
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-05,
    "num_epochs": 3,
    "mixed_precision": true,
    "gradient_checkpointing": true
  }
}


## Create Comparison Table

## Visualization 1: Side-by-Side Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models = ['TF-IDF + LR', 'BERT', 'ClinicalBERT']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# Plot 1: Accuracy
ax = axes[0]
test_acc_values = [tfidf_metrics['test_accuracy'], bert_metrics['test_accuracy'], clinicalbert_metrics['test_accuracy']]
bars = ax.bar(models, test_acc_values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Test Set Accuracy Comparison', fontweight='bold', fontsize=12)
ax.set_ylim([0, 1])
ax.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, test_acc_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# Plot 2: Weighted F1
ax = axes[1]
test_f1_values = [tfidf_metrics['test_weighted_f1'], bert_metrics['test_weighted_f1'], clinicalbert_metrics['test_weighted_f1']]
bars = ax.bar(models, test_f1_values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Weighted F1', fontsize=11)
ax.set_title('Test Set Weighted F1 Comparison', fontweight='bold', fontsize=12)
ax.set_ylim([0, 1])
ax.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, test_f1_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# Plot 3: Macro F1
ax = axes[2]
if 'test_macro_f1' in tfidf_metrics and 'test_macro_f1' in bert_metrics and 'test_macro_f1' in clinicalbert_metrics:
    test_macro_f1_values = [tfidf_metrics['test_macro_f1'], bert_metrics['test_macro_f1'], clinicalbert_metrics['test_macro_f1']]
    bars = ax.bar(models, test_macro_f1_values, color=colors, alpha=0.7, edgecolor='black')
    ax.set_ylabel('Macro F1', fontsize=11)
    ax.set_title('Test Set Macro F1 Comparison', fontweight='bold', fontsize=12)
    ax.set_ylim([0, 1])
    ax.grid(alpha=0.3, axis='y')
    for bar, val in zip(bars, test_macro_f1_values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
else:
    # Plot train vs val vs test for weighted F1
    sets = ['Train', 'Validation', 'Test']
    tfidf_vals = [tfidf_metrics['train_weighted_f1'],
                  tfidf_metrics['val_weighted_f1'],
                  tfidf_metrics['test_weighted_f1']]
    bert_vals = [bert_metrics['train_weighted_f1'],
                 bert_metrics['val_weighted_f1'],
                 bert_metrics['test_weighted_f1']]
    clinicalbert_vals = [clinicalbert_metrics.get('train_weighted_f1', 0),
                         clinicalbert_metrics['val_weighted_f1'],
                         clinicalbert_metrics['test_weighted_f1']]

    x = np.arange(len(sets))
    width = 0.25
    ax.bar(x - width, tfidf_vals, width, label='TF-IDF + LR', color=colors[0], alpha=0.7)
    ax.bar(x,         bert_vals,  width, label='BERT',         color=colors[1], alpha=0.7)
    ax.bar(x + width, clinicalbert_vals, width, label='ClinicalBERT', color=colors[2], alpha=0.7)
    ax.set_ylabel('Weighted F1', fontsize=11)
    ax.set_title('Weighted F1 Across Splits', fontweight='bold', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(sets)
    ax.set_ylim([0, 1])
    ax.legend()
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('comparison_metrics.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Comparison metrics saved as 'comparison_metrics.png'")

## Visualization 2: Train/Val/Test Performance Curves

## Model Characteristics Comparison

## Recommendations and Next Steps